# 04 LoCoMotif Visual Study

## Objective
Study already-computed LoCoMotif outputs, visualize motif intervals when available, audit runtime/failures, and compare LoCoMotif with Matrix Profile for thesis presentation.

## Input files
- `results/motifs/locomotif/locomotif_motif_results.parquet`
- `results/motifs/locomotif/locomotif_evaluation.parquet`
- `results/motifs/locomotif/locomotif_runtime.parquet`
- `results/motifs/locomotif/04_locomotif_failures.parquet` if available
- Matrix Profile summary files for comparison

## Output folder
`reports/study_notebooks/figures/locomotif` and `reports/study_notebooks/tables`.

## Thesis relevance
LoCoMotif complements Matrix Profile by targeting variable/local-constrained interval motifs. This notebook treats LoCoMotif as a controlled subset experiment unless full-scale outputs are present.

## Analysis-only safety
This notebook never imports or calls STUMPY, STUMP/MSTUMP, HMM fitting, LoCoMotif search, or any other expensive experiment algorithm. It only reads saved result files and thesis-scope feature parquet files, then produces derived tables and figures.


In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

NOTEBOOK_DIR = Path.cwd() / "HPC workflow" / "HPC_Regime_and_motif_discovery" / "notebooks" / "study"
if NOTEBOOK_DIR.exists() and str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from study_helpers import *

ensure_study_output_dirs()
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 11,
})

print("Project root:", PROJECT_ROOT)
print("Workflow root:", WORKFLOW_ROOT)
print("Study outputs:", REPORT_ROOT)


## Load LoCoMotif Results


In [ ]:
loco_dir = result_path("motifs", "locomotif")
loco_paths = {
    "motif_results": resolve_existing_file(loco_dir, "locomotif_motif_results.parquet"),
    "evaluation": resolve_existing_file(loco_dir, "locomotif_evaluation.parquet"),
    "runtime": resolve_existing_file(loco_dir, "locomotif_runtime.parquet"),
    "failures": resolve_existing_file(loco_dir, "04_locomotif_failures.parquet"),
}
loco = safe_read_parquet(loco_paths["motif_results"])
loco_eval = safe_read_parquet(loco_paths["evaluation"])
loco_runtime = safe_read_parquet(loco_paths["runtime"])
loco_failures = safe_read_parquet(loco_paths["failures"])
for col in ["motif_start_timestamp", "motif_end_timestamp"]:
    if col in loco.columns:
        loco[col] = pd.to_datetime(loco[col], errors="coerce")


## File Inventory


In [ ]:
fig_roots = [RESULTS_ROOT / "figures", RESULTS_ROOT / "figures_parallel"]
figure_files = []
for root in fig_roots:
    if root.exists():
        figure_files.extend(sorted(root.rglob("04_*")))

inventory = pd.DataFrame([
    {
        "file": name,
        "path": str(path),
        "exists": path.exists(),
        "rows": len({"motif_results": loco, "evaluation": loco_eval, "runtime": loco_runtime, "failures": loco_failures}.get(name, pd.DataFrame())),
        "columns": len({"motif_results": loco, "evaluation": loco_eval, "runtime": loco_runtime, "failures": loco_failures}.get(name, pd.DataFrame()).columns),
    }
    for name, path in loco_paths.items()
] + [{
    "file": "figures_04_prefix",
    "path": "; ".join(str(p) for p in fig_roots),
    "exists": bool(figure_files),
    "rows": len(figure_files),
    "columns": 0,
}])
display_table(inventory)
save_table(inventory, "study_locomotif_file_inventory")


## Scope and Status


In [ ]:
scope_rows = []
for label, df in [("motif_results", loco), ("evaluation", loco_eval), ("runtime", loco_runtime), ("failures", loco_failures)]:
    scope_rows.append({
        "source": label,
        "rows": len(df),
        "assets": ", ".join(sorted(df["asset"].dropna().astype(str).unique())) if "asset" in df.columns and not df.empty else "",
        "frequencies": ", ".join(sorted(df["frequency"].dropna().astype(str).unique())) if "frequency" in df.columns and not df.empty else "",
        "modes": ", ".join(sorted(df["mode"].dropna().astype(str).unique())) if "mode" in df.columns and not df.empty else "",
        "regime_methods": ", ".join(sorted(df["regime_method"].dropna().astype(str).unique())) if "regime_method" in df.columns and not df.empty else "",
        "regime_labels": ", ".join(sorted(df["regime_label"].dropna().astype(str).unique())) if "regime_label" in df.columns and not df.empty else "",
        "feature_sets": ", ".join(sorted(df["feature_set"].dropna().astype(str).unique())) if "feature_set" in df.columns and not df.empty else "",
        "status_counts": json.dumps(df["status"].value_counts(dropna=False).to_dict(), default=str) if "status" in df.columns and not df.empty else "",
    })
scope = pd.DataFrame(scope_rows)
display_table(scope)
save_table(scope, "study_locomotif_scope_table")


## Motif Interval Length Distribution


In [ ]:
length_col = next((c for c in ["motif_length", "interval_length", "length"] if c in loco.columns and pd.api.types.is_numeric_dtype(loco[c])), None)
if loco.empty or not length_col:
    print("No LoCoMotif interval-length rows are available.")
else:
    fig, ax = plt.subplots(figsize=(10, 5))
    loco[length_col].dropna().plot(kind="hist", bins=40, ax=ax, color="#4C78A8")
    ax.set_title("LoCoMotif interval length distribution")
    ax.set_xlabel(length_col)
    ax.set_ylabel("Intervals")
    fig.tight_layout()
    save_fig(fig, "study_locomotif_interval_length_distribution", FIGURE_DIRS["locomotif"])
    plt.show()

    for by in ["mode", "regime_label"]:
        if by in loco.columns:
            groups = [part[length_col].dropna().to_numpy() for _, part in loco.groupby(by, dropna=False)]
            labels = [str(k) for k, _ in loco.groupby(by, dropna=False)]
            if groups:
                fig, ax = plt.subplots(figsize=(10, 5))
                ax.boxplot(groups, labels=labels, showfliers=False)
                ax.set_title(f"LoCoMotif interval length by {by}")
                ax.set_xlabel(by)
                ax.set_ylabel(length_col)
                ax.tick_params(axis="x", rotation=45)
                fig.tight_layout()
                save_fig(fig, f"study_locomotif_interval_length_by_{by}", FIGURE_DIRS["locomotif"])
                plt.show()


## Recurrence and Evaluation


In [ ]:
display_table(loco_eval, 20)
save_table(loco_eval, "study_locomotif_evaluation_raw")

metrics = [c for c in ["number_of_motifs", "recurrence_count", "mean_motif_length", "median_motif_length", "time_split_stability", "cross_regime_overlap"] if c in loco_eval.columns]
if not loco_eval.empty and metrics:
    group = "mode" if "mode" in loco_eval.columns else None
    if group:
        summary = loco_eval.groupby(group, dropna=False)[metrics].agg(["mean", "median", "min", "max"])
        summary.columns = ["_".join(c).strip("_") for c in summary.columns.to_flat_index()]
        summary = summary.reset_index()
        display_table(summary)
        save_table(summary, "study_locomotif_evaluation_by_mode")
        for metric in metrics:
            fig, ax = plt.subplots(figsize=(9, 5))
            loco_eval.groupby(group, dropna=False)[metric].mean().plot(kind="bar", ax=ax, color="#4C78A8")
            ax.set_title(f"LoCoMotif {metric} by mode")
            ax.set_ylabel(metric)
            fig.tight_layout()
            save_fig(fig, f"study_locomotif_{metric}_by_mode", FIGURE_DIRS["locomotif"])
            plt.show()
else:
    print("No populated LoCoMotif evaluation metrics are available.")


## Top LoCoMotif Examples


In [ ]:
def rank_locomotif(df):
    if df.empty:
        return df
    work = df.copy()
    if "motif_score" in work.columns and pd.api.types.is_numeric_dtype(work["motif_score"]):
        return work.sort_values("motif_score", ascending=False)
    for col in ["recurrence_count", "number_of_motifs", "time_split_stability"]:
        if col in work.columns and pd.api.types.is_numeric_dtype(work[col]):
            return work.sort_values(col, ascending=False)
    return work

def loco_top_table(df, name, n=10):
    ranked = rank_locomotif(df).head(n)
    if ranked.empty:
        print(f"No rows for {name}")
        return ranked
    cols = [c for c in [
        "asset", "frequency", "mode", "regime_method", "regime_label", "feature_set",
        "motif_set_rank", "motif_instance_id", "motif_start", "motif_end",
        "motif_start_timestamp", "motif_end_timestamp", "motif_length",
        "motif_score", "motif_set_size", "number_of_motifs", "recurrence_count",
        "mean_motif_length", "runtime_seconds", "status",
    ] if c in ranked.columns]
    out = ranked[cols]
    display_table(out, n)
    save_table(out, name)
    return out

source = loco if not loco.empty else loco_eval
loco_top_table(source[source["mode"].astype(str).eq("agnostic")] if "mode" in source.columns else pd.DataFrame(), "study_locomotif_top10_agnostic")
loco_top_table(source[source["mode"].astype(str).eq("conditioned")] if "mode" in source.columns else pd.DataFrame(), "study_locomotif_top10_conditioned")
for label in ["high_vol", "low_vol"]:
    loco_top_table(source[source["regime_label"].astype(str).eq(label)] if "regime_label" in source.columns else pd.DataFrame(), f"study_locomotif_top10_{label}")


## Interval Illustrations


In [ ]:
def plot_locomotif_timeline(df, asset, frequency, regime_label=None, filename="study_locomotif_timeline"):
    subset = filter_scope(df, asset, frequency)
    if regime_label and "regime_label" in subset.columns:
        subset = subset[subset["regime_label"].astype(str).eq(regime_label)]
    if subset.empty:
        print(f"No LoCoMotif intervals for {asset} {frequency} {regime_label or ''}")
        return
    feature = load_feature_data(asset, frequency)
    ts = timestamp_column(feature)
    fig, ax = plt.subplots(figsize=(14, 5))
    if not feature.empty and ts and "close" in feature.columns:
        sampled = feature[[ts, "close"]].dropna().sort_values(ts)
        if len(sampled) > 8000:
            sampled = sampled.iloc[:: int(np.ceil(len(sampled) / 8000))]
        ax.plot(sampled[ts], sampled["close"], color="0.75", linewidth=0.8, label="close")
    draw = subset.head(50)
    for j, (_, row) in enumerate(draw.iterrows()):
        start = row.get("motif_start_timestamp")
        end = row.get("motif_end_timestamp")
        if pd.notna(start) and pd.notna(end):
            ax.axvspan(pd.to_datetime(start), pd.to_datetime(end), color=plt.cm.tab10(j % 10), alpha=0.2)
    ax.set_title(f"LoCoMotif intervals - {asset} {frequency} {regime_label or 'all'}")
    ax.set_xlabel("Timestamp")
    ax.set_ylabel("Close price or interval spans")
    fig.autofmt_xdate()
    fig.tight_layout()
    save_fig(fig, filename, FIGURE_DIRS["locomotif"])
    plt.show()

plot_locomotif_timeline(loco, "BTCUSDT", "15m", None, "study_locomotif_BTCUSDT_15m_agnostic_intervals")
plot_locomotif_timeline(loco, "BTCUSDT", "15m", "high_vol", "study_locomotif_BTCUSDT_15m_high_vol_intervals")
plot_locomotif_timeline(loco, "BTCUSDT", "15m", "low_vol", "study_locomotif_BTCUSDT_15m_low_vol_intervals")

if figure_files:
    fig_table = pd.DataFrame({"figure_path": [str(p) for p in figure_files]})
    display_table(fig_table, 20)
    save_table(fig_table, "study_locomotif_existing_figure_inventory")
else:
    print("No existing LoCoMotif figures with prefix 04_ were found.")


## Runtime and Failure Audit


In [ ]:
display_table(loco_runtime, 30)
save_table(loco_runtime, "study_locomotif_runtime_raw")
if "status" in loco_runtime.columns and not loco_runtime.empty:
    status_counts = loco_runtime["status"].value_counts(dropna=False).rename_axis("status").reset_index(name="rows")
    display_table(status_counts)
    save_table(status_counts, "study_locomotif_runtime_status_counts")

if not loco_paths["failures"].exists():
    print("LoCoMotif failure file is not available.")
elif loco_failures.empty:
    print("LoCoMotif failure file exists and contains no rows; no internal failures recorded in that file.")
else:
    display_table(loco_failures, 30)
    save_table(loco_failures, "study_locomotif_failures")

print("LoCoMotif is reported as a controlled subset experiment rather than a full-scale benchmark.")


## LoCoMotif vs Matrix Profile


In [ ]:
mp_dir = result_path("motifs", "matrix_profile")
mp_results = safe_read_parquet(resolve_existing_file(mp_dir, "matrix_profile_motif_results.parquet"))
mp_eval = safe_read_parquet(resolve_existing_file(mp_dir, "matrix_profile_evaluation.parquet"))
mp_runtime = safe_read_parquet(resolve_existing_file(mp_dir, "matrix_profile_runtime.parquet"))

comparison = pd.DataFrame([
    {"Aspect": "Motif type", "Matrix Profile": "fixed-length subsequences", "LoCoMotif": "variable/local-constrained intervals"},
    {"Aspect": "Scale completed", "Matrix Profile": "full controlled benchmark", "LoCoMotif": "controlled subset"},
    {"Aspect": "Result rows", "Matrix Profile": len(mp_results), "LoCoMotif": len(loco)},
    {"Aspect": "Evaluation rows", "Matrix Profile": len(mp_eval), "LoCoMotif": len(loco_eval)},
    {"Aspect": "Runtime rows", "Matrix Profile": len(mp_runtime), "LoCoMotif": len(loco_runtime)},
    {"Aspect": "Strength", "Matrix Profile": "scalable baseline", "LoCoMotif": "flexible variable-length motif structure"},
    {"Aspect": "Limitation", "Matrix Profile": "fixed-length windows", "LoCoMotif": "more computationally constrained"},
])
display_table(comparison, 10)
save_table(comparison, "study_locomotif_vs_matrix_profile_comparison")


## Key findings
LoCoMotif adds a variable/local-constrained interval perspective when populated interval or evaluation rows are available. Runtime and failure tables determine whether the run should be presented as completed, partial, or empty.

## Thesis-safe interpretation
LoCoMotif should be presented as a complementary controlled subset experiment unless the saved files show full-scale coverage. It supports the Matrix Profile results by testing a more flexible motif formulation, but it should not be framed as a like-for-like scalability benchmark unless the result files support that claim.

## Limitations
This notebook does not rerun LoCoMotif. Empty motif or evaluation outputs mean no motif-quality numerical claims can be made from LoCoMotif rows. Existing figures are inventoried but not treated as new evidence unless their source files are present.

## Recommended figures for thesis
- LoCoMotif interval timeline when interval rows are available
- Runtime status counts
- LoCoMotif vs Matrix Profile comparison table
- Existing selected `04_` figures from the controlled subset run
